# Employee Attrition Prediction
### A complete end-to-end Machine Learning Classification Project

**Dataset:** IBM HR Analytics Employee Attrition & Performance (1470 employees, 35 columns)

**Goal:** Predict whether an employee will leave the company (`Attrition`: Yes/No) based on demographic, job, and satisfaction related features.

This notebook follows the same structured workflow as a typical **Loan Approval Prediction** project:
1. Import Libraries
2. Load & Explore the Dataset
3. Exploratory Data Analysis (EDA)
4. Data Cleaning & Preprocessing (handling categorical data, encoding, scaling)
5. Train-Test Split
6. Model Building (multiple classifiers)
7. Model Evaluation & Comparison
8. Feature Importance
9. Conclusion & Business Insights

> Run the cells in order (Shift + Enter). Everything needed is self-contained — the dataset is fetched automatically if the local CSV is not found.

## 1. Import Required Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# Preprocessing & model selection
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Evaluation metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

## 2. Load the Dataset

The dataset used is the well-known **IBM HR Analytics Employee Attrition & Performance** dataset.
The code below first looks for a local file `employee_attrition.csv` in the same folder as this notebook.
If it isn't found, it automatically downloads it from a public GitHub mirror.

In [ ]:
import os

local_path = "employee_attrition.csv"
url = "https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
    print("Loaded dataset from local file.")
else:
    df = pd.read_csv(url)
    df.to_csv(local_path, index=False)
    print("Downloaded dataset and saved a local copy.")

print("Shape of dataset:", df.shape)
df.head()

## 3. Basic Data Exploration

In [ ]:
# Structure of the dataset
df.info()

In [ ]:
# Statistical summary of numerical columns
df.describe().T

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Total missing values in dataset:", missing.sum())
missing[missing > 0]

In [ ]:
# Check for duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
# Target variable distribution
print(df['Attrition'].value_counts())
print(df['Attrition'].value_counts(normalize=True) * 100)

## 4. Exploratory Data Analysis (EDA)

We look at how the target variable (`Attrition`) relates to other important features, similar to how in
loan-approval projects we check how `Loan_Status` relates to income, credit history, etc.

In [ ]:
# Attrition class distribution (imbalance check)
plt.figure(figsize=(5,4))
sns.countplot(x='Attrition', data=df, palette='Set2')
plt.title("Attrition Distribution (Target Variable)")
plt.xlabel("Attrition")
plt.ylabel("Count")
plt.show()

In [ ]:
# Age distribution by Attrition
plt.figure(figsize=(7,5))
sns.histplot(data=df, x='Age', hue='Attrition', kde=True, multiple='stack', palette='Set2')
plt.title("Age Distribution by Attrition")
plt.show()

In [ ]:
# Monthly Income vs Attrition
plt.figure(figsize=(7,5))
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df, palette='Set2')
plt.title("Monthly Income vs Attrition")
plt.show()

In [ ]:
# OverTime vs Attrition
plt.figure(figsize=(6,4))
sns.countplot(x='OverTime', hue='Attrition', data=df, palette='Set2')
plt.title("Attrition by OverTime Status")
plt.show()

In [ ]:
# JobSatisfaction vs Attrition
plt.figure(figsize=(6,4))
sns.countplot(x='JobSatisfaction', hue='Attrition', data=df, palette='Set2')
plt.title("Attrition by Job Satisfaction Level")
plt.show()

In [ ]:
# Department wise attrition
plt.figure(figsize=(7,4))
sns.countplot(x='Department', hue='Attrition', data=df, palette='Set2')
plt.title("Attrition by Department")
plt.xticks(rotation=15)
plt.show()

In [ ]:
# Correlation heatmap for numerical features
plt.figure(figsize=(14,10))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), cmap='coolwarm', linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()

## 5. Data Preprocessing

Just like in the loan approval project we:
- Drop irrelevant / constant columns
- Encode categorical variables
- Encode the target variable
- Scale numerical features
- Split the data into train and test sets

In [ ]:
# Drop columns that provide no useful information (constant or identifiers)
cols_to_drop = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print("Remaining columns:", df.shape[1])
df.head()

In [ ]:
# Encode the target variable: Yes -> 1, No -> 0
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
df['Attrition'].value_counts()

In [ ]:
# Identify categorical columns to encode
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", categorical_cols)

In [ ]:
# Label Encode categorical columns (same approach used for loan approval categorical fields)
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

df.head()

In [ ]:
# Separate features (X) and target (y)
X = df.drop('Attrition', axis=1)
y = df['Attrition']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Feature scaling - important for Logistic Regression, KNN, SVM
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

## 7. Model Building

We train several classification models — exactly the same set of algorithms typically used
for the Loan Approval problem — and compare their performance:

- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- Support Vector Machine (SVM)
- K-Nearest Neighbors (KNN)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan

    results.append([name, acc, prec, rec, f1, auc])
    print(f"{name} trained. Accuracy: {acc:.4f}")

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"])
results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

## 8. Detailed Evaluation of the Best Model

We take a closer look at the model with the best overall performance (based on the comparison table above),
including a confusion matrix, classification report, and ROC curve.

In [ ]:
best_model_name = results_df.sort_values(by="Accuracy", ascending=False).iloc[0]["Model"]
best_model = models[best_model_name]
print("Best performing model:", best_model_name)

y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Attrition', 'Attrition'],
            yticklabels=['No Attrition', 'Attrition'])
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# ROC Curve for all models that support probability prediction
plt.figure(figsize=(7,6))
for name, model in models.items():
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.2f})")

plt.plot([0,1],[0,1],'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

## 9. Model Comparison Chart

In [ ]:
plt.figure(figsize=(9,5))
results_melted = results_df.melt(id_vars="Model", value_vars=["Accuracy", "Precision", "Recall", "F1-Score"])
sns.barplot(data=results_melted, x="Model", y="value", hue="variable")
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 10. Feature Importance (Random Forest)

Understanding which features contribute the most to predicting attrition helps HR teams
take targeted, data-driven retention actions.

In [ ]:
rf_model = models["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(9,8))
sns.barplot(x=importances.values[:15], y=importances.index[:15], palette='viridis')
plt.title("Top 15 Most Important Features (Random Forest)")
plt.xlabel("Importance Score")
plt.show()

importances.head(15)

## 11. Hyperparameter Tuning (Optional Improvement)

A quick `GridSearchCV` example on Random Forest to squeeze out extra performance —
the same technique commonly applied at the end of the loan approval notebook.

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy: {:.4f}".format(grid_search.best_score_))

tuned_model = grid_search.best_estimator_
tuned_pred = tuned_model.predict(X_test)
print("Test Accuracy after tuning: {:.4f}".format(accuracy_score(y_test, tuned_pred)))

## 12. Conclusion & Business Insights

- The dataset is **imbalanced** (roughly 84% "No Attrition" vs 16% "Attrition"), so metrics like
  **Precision, Recall, F1-score, and ROC-AUC** matter more than plain accuracy.
- Key drivers of attrition typically include: **OverTime, MonthlyIncome, Age, TotalWorkingYears,
  YearsAtCompany, JobSatisfaction, and DistanceFromHome** (see the feature importance chart above —
  exact ranking may vary slightly run to run).
- **Ensemble models (Random Forest / Gradient Boosting)** generally outperform simpler models like
  plain Logistic Regression on this dataset, similar to what is usually observed on the loan approval dataset.
- **HR Recommendation:** Focus retention efforts on employees who work overtime frequently, have lower
  income relative to role/level, and report low job satisfaction — these groups show the highest attrition risk.

### Concepts practiced in this notebook (same as Loan Approval project)
- Handling a real-world tabular dataset with mixed categorical/numerical features
- Label Encoding & Feature Scaling
- Train/Test split with stratification (to preserve class ratio)
- Training & comparing multiple classification algorithms
- Evaluation using Accuracy, Precision, Recall, F1-score, ROC-AUC, Confusion Matrix
- Feature importance interpretation
- Hyperparameter tuning with GridSearchCV

**Good luck with your exam tomorrow!**